# Dialogues protocolisés : inquiry et persuasion (Walton–Krabbe) — jumeau C#

Jumeau C# de [04-Dialogues-Protocolises-Python](./Argumentation-04-Dialogues-Protocolises-Python.ipynb) :
mêmes neuf actes de parole, mêmes tables de transitions, mêmes fixtures partagées
(`data/dialogue_protocols_examples.json`) — portées ici en **C# BCL natif** pour la
partie protocoles, et exécutées sur le **vrai Tweety via IKVM** pour la partie
grounding.

**Différence de moteur déclarée.** Le jumeau Python exécute le dialogue de grounding
via un pont `jpype` vers le JVM (carnet `00-Setup` requis). Ce jumeau C# charge
directement le shade IKVM du dépôt (`org.tweetyproject.tweety-dung.dll`, compilé
par `dotnet-build/build-TweetyDungShade.csproj`) : même sémantique Dung, même
`SimpleGroundedReasoner`, **zéro JVM**. Les deux verdicts attendus sont inchangés :
`accepted` sur la dessalinisation, `rejected` sur la variante carbone.

**Série Argumentation** | | Précédent : [03d-ADF-Semantiques-Python](./Argumentation-03d-ADF-Semantiques-Python.ipynb) | Suivant : [04b-Knowledge-Base-Python](./Argumentation-04b-Knowledge-Base-Python.ipynb) | [↑ Série Argumentation](README.md) | [jumeau Python](Argumentation-04-Dialogues-Protocolises-Python.ipynb)

## Pourquoi protocoliser un dialogue ?

Walton et Krabbe (*Commitment in Dialogue*, 1995) typent les dialogues par leur but :
chercher de l'information, enquêter, persuader, négocier, délibérer, ou simplement
s'affronter. Un dialogue laissé libre ne sait dire ni qu'il a mal tourné, ni qu'il
est fini : une conversation qui tourne en rond et une conversation qui conclut se
ressemblent, vues de l'extérieur, tant qu'aucune règle ne les distingue.

Protocoliser, c'est rendre ces deux jugements **mécaniques** :

1. **Un coup illégal a une position.** La table de transitions dit, pour chaque
   acte, quels actes peuvent légitimement suivre. Une contestation — « tu n'avais
   pas le droit de concéder ici » — se tranche en consultant la table, pas en
   argumentant sur les intentions.
2. **La fin est une condition, pas un sentiment.** Compréhension mutuelle,
   capitulation, impasse, longueur maximale : le protocole énumère ses façons de
   mourir, et coupe lui-même les dialogues qui tournent en rond.
3. **Le dialogue devient reproductible.** Deux traces identiques reçoivent le même
   verdict : le dialogue devient l'objet d'une expérience, pas seulement d'une
   conversation.

Le registre Walton–Krabbe déclare six types de dialogue ; deux seulement ont une
machine. Ce n'est pas un défaut à corriger mais un état de la recherche : une
typologie nomme plus qu'elle ne construit, et l'écart entre les deux est en soi
instructif — c'est la première chose que mesure ce carnet. Le port C# est **fidèle**
aux tables du jumeau Python, angles morts compris : la période 3 invisible et la
condition de double-retraction inatteignable y sont mesurées exactement comme en Python.


In [1]:
#r "nuget: IKVM, 8.15.0"
#r "nuget: IKVM.Image, 8.15.0"
#r "../Tweety/org.tweetyproject.tweety-dung.dll"
// --- Setup : utilitaires d'affichage + charge du shade Tweety via IKVM ---
using System;
using System.Collections.Generic;
using System.IO;
using System.Linq;
using System.Runtime.InteropServices;
using System.Text;

// Helper d'affichage : en headless, Console.WriteLine peut etre avale.
static void Show(object o)
{
    try { display(o); } catch { Console.WriteLine(o); }
}
// Surcharge etiquette+valeur (les etiquettes de ce carnet finissent par ": ").
static void Show(string label, object value) => Show(label + value);

// Formateur de tableau compact (memorandum du jumeau Tweety-08-CSharp).
static string Table(IEnumerable<string> headers, IEnumerable<IEnumerable<string>> rows)
{
    var allRows = rows.ToList();
    var widths = headers.Select((h, i) =>
        Math.Max(h.Length, allRows.Count > 0 ? allRows.Max(r => r.ElementAtOrDefault(i)?.Length ?? 0) : 0)).ToList();
    var sb = new StringBuilder();
    sb.AppendLine(string.Join(" | ", headers.Select((h, i) => h.PadRight(widths[i]))));
    sb.AppendLine(string.Join("-+-", widths.Select(w => new string('-', w))));
    foreach (var row in allRows)
        sb.AppendLine(string.Join(" | ", row.Select((c, i) => c.PadRight(i < widths.Count ? widths[i] : c.Length))));
    return sb.ToString();
}

// Domicile IKVM (pattern du jumeau Tweety-03-Dung) : le paquet nuget IKVM.Image
// dissimule les ressources du runtime sous $NUGET_PACKAGES/ikvm.image*/... ; le
// JVM embarque des assemblies ombrées les cherche a un « home » declare par
// AppContext. Sans cette cellule, premiere instanciation = « Could not locate
// ikvm home path ».
string ikvmVer = "8.15.0";
string ikvmRid = (OperatingSystem.IsWindows() ? "win" : OperatingSystem.IsMacOS() ? "osx" : "linux")
    + "-" + RuntimeInformation.ProcessArchitecture.ToString().ToLowerInvariant();
string nugetRoot = Environment.GetEnvironmentVariable("NUGET_PACKAGES")
    ?? Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.UserProfile), ".nuget", "packages");
string ikvmBaseAny = Path.Combine(nugetRoot, "ikvm.image", ikvmVer, "ikvm", "any", "any");
string ikvmArchDir = Path.Combine(nugetRoot, "ikvm.image.runtime." + ikvmRid, ikvmVer, "ikvm", "any", ikvmRid);
string ikvmHome = Path.Combine(Path.GetTempPath(), "ikvm-home-" + ikvmVer + "-" + ikvmRid);
void IkvmCopyMerge(string src, string dst)
{
    foreach (var d in Directory.GetDirectories(src, "*", SearchOption.AllDirectories))
        Directory.CreateDirectory(d.Replace(src, dst));
    foreach (var f in Directory.GetFiles(src, "*", SearchOption.AllDirectories))
    {
        var t = f.Replace(src, dst);
        Directory.CreateDirectory(Path.GetDirectoryName(t));
        File.Copy(f, t, overwrite: true);
    }
}
if (Directory.Exists(ikvmBaseAny) && Directory.Exists(ikvmArchDir))
{
    Directory.CreateDirectory(ikvmHome);
    IkvmCopyMerge(ikvmBaseAny, ikvmHome);
    IkvmCopyMerge(ikvmArchDir, ikvmHome);
}
AppContext.SetData("IKVM.Home", ikvmHome);
bool ikvmReady = File.Exists(Path.Combine(ikvmHome, "lib", "tzdb.dat"));
Show("Setup OK — domicile IKVM " + ikvmRid + " (" + (ikvmReady ? "OK" : "MISSING") + "), "
    + "shade en reference de compilation.");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages IKVM, 8.15.0 IKVM.Image, 8.15.0

Setup OK — domicile IKVM win-x64 (OK), shade en reference de compilation.

In [2]:
// Verification du shade — Constat 3 (#5039), dans une cellule SEPARÉE du #r :
// le bind de compilation charge l'assembly avant l'exécution des instructions de
// la cellule, donc un cctor IKVM déclenché dans la cellule de setup ne verrait
// pas le domicile déclaré ci-dessus (« Could not locate ikvm home path »).
// AppDomain.GetAssemblies() est un oracle faux (#r = reference de compilation,
// pas un chargement AppDomain) : le vrai test, c'est d'instancier. #r manquant
// = CS0246 à la compilation ; runtime rompu = échec explicite ici. Aucun repli
// silencieux (anti-theatre, #2137).
var probe = new org.tweetyproject.arg.dung.syntax.DungTheory();
if (!ikvmReady)
    throw new InvalidOperationException(
        "Domicile IKVM incomplet (lib/tzdb.dat absent) — le runtime embarque n'est pas " +
        "pret. Verifier le cache nuget (paquets ikvm.image 8.15.0). Mode degrade " +
        "INTERDIT (anti-theatre, #2137).");
Show("Shade verifie par instanciation : " + probe.GetType().Assembly.GetName().Name
    + " — vraie sémantique Dung de Tweety, 0 JVM.");

Shade verifie par instanciation : org.tweetyproject.tweety-dung — vraie sémantique Dung de Tweety, 0 JVM.

In [3]:
// --- Port BCL de dialogue_protocols.py : machine a etats sur actes de langage ---
// Portage fidèle : memes tables de transitions, memes conditions de terminaison,
// y compris leurs angles morts documentes (periode 3 invisible, double-retract mort).

public enum DialogueType
{
    INFORMATION_SEEKING, INQUIRY, PERSUASION,
    NEGOTIATION, DELIBERATION, ERISTIC
}

public enum SpeechAct
{
    CLAIM, QUESTION, CHALLENGE, ARGUE, CONCEDE,
    RETRACT, SUPPORT, REFUTE, UNDERSTAND
}

public class DialogueMove
{
    public string Speaker { get; }
    public SpeechAct Act { get; }
    public string Content { get; }
    public string? Target { get; }
    public DialogueMove(string speaker, SpeechAct act, string content = "", string? target = null)
    { Speaker = speaker; Act = act; Content = content; Target = target; }
}

public class TermCond
{
    public string Name { get; }
    public Func<List<DialogueMove>, bool> Fn { get; }
    public TermCond(string name, Func<List<DialogueMove>, bool> fn) { Name = name; Fn = fn; }
}

public class TraceVerdict
{
    public bool Valid { get; init; }
    public int? FirstInvalid { get; init; }
    public bool Terminated { get; init; }
    public string? Reason { get; init; }
}

public abstract class DialogueProtocol
{
    public string Name { get; protected set; } = "abstract";
    public Dictionary<SpeechAct, List<SpeechAct>> AllowedTransitions { get; protected set; } = new();
    public List<TermCond> TerminationConditions { get; protected set; } = new();

    protected abstract void SetupProtocol();

    public DialogueProtocol() => SetupProtocol();

    public bool IsValidMove(SpeechAct current, SpeechAct next)
        => AllowedTransitions.TryGetValue(current, out var succ) && succ.Contains(next);

    public bool IsTerminalState(List<DialogueMove> history)
        => TerminationConditions.Any(c => c.Fn(history));

    public string? TerminationReason(List<DialogueMove> history)
        => TerminationConditions.FirstOrDefault(c => c.Fn(history))?.Name;

    public List<SpeechAct> GetAllowedResponses(SpeechAct lastAct)
        => AllowedTransitions.TryGetValue(lastAct, out var succ) ? succ : new List<SpeechAct>();

    public TraceVerdict ValidateTrace(List<DialogueMove> moves)
    {
        int? firstInvalid = null;
        for (int i = 1; i < moves.Count; i++)
            if (!IsValidMove(moves[i - 1].Act, moves[i].Act)) { firstInvalid = i; break; }
        return new TraceVerdict
        {
            Valid = firstInvalid is null,
            FirstInvalid = firstInvalid,
            Terminated = IsTerminalState(moves),
            Reason = TerminationReason(moves),
        };
    }
}


(22,18): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(23,83): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(39,18): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(58,18): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



In [4]:
// Les deux machines : tables portees a l'identique du jumeau Python.
public class InquiryProtocol : DialogueProtocol
{
    protected override void SetupProtocol()
    {
        Name = "inquiry";
        AllowedTransitions = new Dictionary<SpeechAct, List<SpeechAct>>
        {
            [SpeechAct.QUESTION]   = new() { SpeechAct.CLAIM, SpeechAct.ARGUE, SpeechAct.QUESTION, SpeechAct.SUPPORT },
            [SpeechAct.CLAIM]      = new() { SpeechAct.SUPPORT, SpeechAct.CHALLENGE, SpeechAct.QUESTION, SpeechAct.UNDERSTAND },
            [SpeechAct.SUPPORT]    = new() { SpeechAct.QUESTION, SpeechAct.UNDERSTAND, SpeechAct.CHALLENGE },
            [SpeechAct.CHALLENGE]  = new() { SpeechAct.ARGUE, SpeechAct.SUPPORT, SpeechAct.QUESTION },
            [SpeechAct.ARGUE]      = new() { SpeechAct.CHALLENGE, SpeechAct.UNDERSTAND, SpeechAct.QUESTION, SpeechAct.SUPPORT },
            [SpeechAct.UNDERSTAND] = new() { SpeechAct.QUESTION, SpeechAct.UNDERSTAND, SpeechAct.CLAIM },
            [SpeechAct.REFUTE]     = new() { SpeechAct.ARGUE, SpeechAct.QUESTION },
            [SpeechAct.CONCEDE]    = new() { SpeechAct.QUESTION, SpeechAct.UNDERSTAND },
        };
        TerminationConditions = new List<TermCond>
        {
            new("_term_mutual_understanding", h => h.Count >= 3
                && h.TakeLast(3).All(m => m.Act is SpeechAct.UNDERSTAND or SpeechAct.CONCEDE)),
            new("_term_length", h => h.Count > 25),
            new("_term_converged", h => h.Count >= 4
                && h[^1].Act == SpeechAct.UNDERSTAND && h[^2].Act == SpeechAct.UNDERSTAND),
            new("_term_loop", h => h.Count >= 6
                && (h[^6].Act, h[^5].Act) == (h[^4].Act, h[^3].Act)
                && (h[^4].Act, h[^3].Act) == (h[^2].Act, h[^1].Act)),
        };
    }
}

public class PersuasionProtocol : DialogueProtocol
{
    protected override void SetupProtocol()
    {
        Name = "persuasion";
        AllowedTransitions = new Dictionary<SpeechAct, List<SpeechAct>>
        {
            [SpeechAct.CLAIM]      = new() { SpeechAct.CHALLENGE, SpeechAct.CONCEDE, SpeechAct.QUESTION, SpeechAct.SUPPORT },
            [SpeechAct.CHALLENGE]  = new() { SpeechAct.ARGUE, SpeechAct.RETRACT, SpeechAct.SUPPORT },
            [SpeechAct.ARGUE]      = new() { SpeechAct.CHALLENGE, SpeechAct.CONCEDE, SpeechAct.REFUTE, SpeechAct.SUPPORT },
            [SpeechAct.QUESTION]   = new() { SpeechAct.CLAIM, SpeechAct.ARGUE, SpeechAct.SUPPORT },
            [SpeechAct.REFUTE]     = new() { SpeechAct.ARGUE, SpeechAct.CONCEDE, SpeechAct.CHALLENGE },
            [SpeechAct.SUPPORT]    = new() { SpeechAct.UNDERSTAND, SpeechAct.CHALLENGE, SpeechAct.QUESTION },
            [SpeechAct.CONCEDE]    = new() { SpeechAct.CLAIM, SpeechAct.QUESTION },
            [SpeechAct.RETRACT]    = new() { SpeechAct.CLAIM, SpeechAct.QUESTION },
            [SpeechAct.UNDERSTAND] = new() { SpeechAct.CLAIM, SpeechAct.QUESTION },
        };
        TerminationConditions = new List<TermCond>
        {
            new("_term_concede", h => h.Count > 0 && h[^1].Act == SpeechAct.CONCEDE),
            new("_term_length", h => h.Count > 30),
            new("_term_double_retract", h => h.Count > 2
                && h[^1].Act == SpeechAct.RETRACT && h[^2].Act == SpeechAct.RETRACT),
        };
    }
}

var protos = new Dictionary<string, DialogueProtocol>
{
    ["inquiry"] = new InquiryProtocol(),
    ["persuasion"] = new PersuasionProtocol(),
};
var inq = (InquiryProtocol)protos["inquiry"];
var per = (PersuasionProtocol)protos["persuasion"];

// Inventaire de l'organe (equivalent de protocol_counts) :
int CountTransitions(DialogueProtocol p) => p.AllowedTransitions.Values.Sum(v => v.Count);
Show("Types Walton-Krabbe au registre :", Enum.GetValues(typeof(DialogueType)).Length);
Show("Actes de parole formalises     :", Enum.GetValues(typeof(SpeechAct)).Length);
Show("Protocoles implantes (machine) :", string.Join(", ", protos.Keys.OrderBy(k => k)));
Show("Transitions inquiry            :", CountTransitions(inq));
Show("Transitions persuasion         :", CountTransitions(per));

Types Walton-Krabbe au registre :6

Actes de parole formalises     :9

Protocoles implantes (machine) :inquiry, persuasion

Transitions inquiry            :25

Transitions persuasion         :26

### Lecture du compte

Le registre déclare six types — `INFORMATION_SEEKING`, `INQUIRY`, `PERSUASION`,
`NEGOTIATION`, `DELIBERATION`, `ERISTIC` — mais seuls `inquiry` et `persuasion`
ont une machine : quatre types sont des noms sans protocole. Les deux machines
sont d'une taille quasi identique (25 transitions contre 26) alors que leurs
natures s'opposent : la différence entre coopératif et adversarial n'est pas une
question de volume mais de *composition* — c'est ce que lira le tableau croisé
de fin de carnet, ligne par ligne.


## Les neuf actes de parole

La pragmatique linguistique classe les actes de langage par ce qu'ils *font*
plutôt que par ce qu'ils disent. Le registre des actes en formalise neuf, que
l'on peut grouper en quatre familles :

| Famille | Actes | Effet sur l'engagement |
|---|---|---|
| Assertifs | `CLAIM`, `SUPPORT` | le locuteur s'engage sur un contenu |
| Directifs | `QUESTION`, `CHALLENGE` | le locuteur exige quelque chose de l'interlocuteur — une réponse, une justification |
| Argumentatifs | `ARGUE`, `REFUTE` | le locuteur attaque ou défend un contenu par des raisons |
| Conclusifs | `CONCEDE`, `RETRACT`, `UNDERSTAND` | le locuteur clôt un point : il cède, retire sa parole, ou déclare avoir compris |

La donnée que manipule la machine n'est ni la phrase ni sa syntaxe : c'est l'acte.
Deux phrases différentes qui contestent — « pourquoi cela ? » et « sur quoi repose
votre chiffre ? » — sont le même coup de `CHALLENGE`. C'est ce nivellement qui rend
l'état d'un dialogue dénombrable : la machine ne compte pas des mots, elle compte
des coups.


In [5]:
var toConcedeInq = Enum.GetValues(typeof(SpeechAct)).Cast<SpeechAct>()
    .Where(a => inq.AllowedTransitions.TryGetValue(a, out var s) && s.Contains(SpeechAct.CONCEDE))
    .Select(a => a.ToString()).ToList();
var toConcedePer = Enum.GetValues(typeof(SpeechAct)).Cast<SpeechAct>()
    .Where(a => per.AllowedTransitions.TryGetValue(a, out var s) && s.Contains(SpeechAct.CONCEDE))
    .Select(a => a.ToString()).ToList();

Show("Apres QUESTION, l'inquiry autorise :", string.Join(", ", inq.GetAllowedResponses(SpeechAct.QUESTION)));
Show("Apres QUESTION, la persuasion autorise :", string.Join(", ", per.GetAllowedResponses(SpeechAct.QUESTION)));
Console.WriteLine();
Show("Actes menant a CONCEDE en inquiry :", toConcedeInq.Count > 0 ? string.Join(", ", toConcedeInq) : "aucun");
Show("Actes menant a CONCEDE en persuasion :", string.Join(", ", toConcedePer));

Apres QUESTION, l'inquiry autorise :CLAIM, ARGUE, QUESTION, SUPPORT

Apres QUESTION, la persuasion autorise :CLAIM, ARGUE, SUPPORT

Actes menant a CONCEDE en inquiry :aucun

Actes menant a CONCEDE en persuasion :CLAIM, ARGUE, REFUTE

### Lecture : deux tables, deux mondes

Après `QUESTION`, l'inquiry autorise… `QUESTION` à nouveau : relancer la
clarification est légitime dans une enquête. La persuasion l'exclut — après une
question, on répond (`CLAIM`, `ARGUE`, `SUPPORT`), on ne questionne pas.

Plus frappant : `CONCEDE` n'est atteignable par **aucun** acte en inquiry —
l'acte de capitulation existe dans l'énumération des neuf, mais aucune transition
n'y mène : c'est un acte mort dans ce protocole. En persuasion, trois actes y
mènent (`CLAIM`, `ARGUE`, `REFUTE`). La capitulation est une porte qui n'existe
que dans le monde adversarial.


## Le verdict attendu : la fixture partagée

Les tables ci-dessus disent ce que le moteur **fait** ; la fixture partagée dit ce
qu'il **doit** faire. Neuf transitions et sept historiques y sont consignés avec
leur verdict attendu et sa raison — c'est la source de vérité commune du carnet de
référence et de la garde aller-retour `tests/test_dialogue_protocols.py`. Passer
les tables au crible de la fixture, c'est séparer la spécification de
l'implémentation : si un jour le moteur diverge, c'est ici que le désaccord aura
un lieu. Le jumeau C# lit **le même fichier** `data/dialogue_protocols_examples.json`
que le jumeau Python : une seule spécification, deux moteurs.


In [6]:
// Les 9 transitions de la fixture : IsValidMove doit rendre le verdict attendu.
using System.Text.Json;

var examples = JsonDocument.Parse(File.ReadAllText("data/dialogue_protocols_examples.json"));
int okCount = 0;
foreach (var ex in examples.RootElement.GetProperty("transitions").EnumerateArray())
{
    var dialogue = ex.GetProperty("dialogue").GetString()!;
    var from = Enum.Parse<SpeechAct>(ex.GetProperty("from").GetString()!);
    var to = Enum.Parse<SpeechAct>(ex.GetProperty("to").GetString()!);
    var allowed = ex.GetProperty("allowed").GetBoolean();
    var got = protos[dialogue].IsValidMove(from, to);
    var marque = got == allowed ? "OK " : "ECART";
    Show($"{marque} {dialogue,-11} {from} -> {to,-11} {(got ? "autorisé" : "interdit")}");
    Show($"    {ex.GetProperty("why").GetString()}");
    if (got == allowed) okCount++;
}
Show($"{okCount}/9 transitions rendent le verdict attendu : {okCount == 9}");

OK  inquiry     QUESTION -> CLAIM       autorisé

    En inquiry, une question peut recevoir une affirmation directe.

OK  inquiry     CLAIM -> CONCEDE     interdit

    Asymétrie clé : en inquiry on ne concède pas une affirmation — on la soutient, la conteste ou la questionne.

OK  inquiry     CHALLENGE -> ARGUE       autorisé

    Une contestation appelle un argument.

... (9 outputs left out)

    Une question n'appelle pas une contestation directe — d'abord une réponse (claim/argue/support).

OK  persuasion  CONCEDE -> ARGUE       interdit

    Après une concession, on reformule une position ou on questionne, on ne relance pas un argument.

9/9 transitions rendent le verdict attendu : True

### Lecture : la spécification a un fichier

Chaque ligne porte son verdict **et sa raison** — l'asymétrie la plus instructive
est `CLAIM -> CONCEDE` : interdit en inquiry (« on ne concède pas une affirmation
qu'on cherche à établir ensemble »), autorisé en persuasion (« concéder est la
fin normale du débat »). Le même couple d'actes, deux mondes : ce n'est pas le
moteur qui le décide, c'est la fixture qui l'exige.


In [7]:
// Les 7 historiques de la fixture : IsTerminalState doit rendre le verdict attendu.
List<DialogueMove> Historique(string[] acts) =>
    acts.Select((a, i) => new DialogueMove($"loc{i % 2}", Enum.Parse<SpeechAct>(a), "-"))
        .ToList();

int okT = 0;
foreach (var ex in examples.RootElement.GetProperty("terminations").EnumerateArray())
{
    var dialogue = ex.GetProperty("dialogue").GetString()!;
    var acts = new List<string>();
    foreach (var a in ex.GetProperty("acts").EnumerateArray()) acts.Add(a.GetString()!);
    var got = protos[dialogue].IsTerminalState(Historique(acts.ToArray()));
    var expected = ex.GetProperty("terminal").GetBoolean();
    var marque = got == expected ? "OK " : "ECART";
    Show($"{marque} {dialogue,-11} {(got ? "TERMINAL" : "en cours"),-9} {string.Join(" -> ", acts)}");
    Show($"    {ex.GetProperty("why").GetString()}");
    if (got == expected) okT++;
}
Show($"{okT}/7 historiques rendent le verdict attendu : {okT == 7}");

OK  inquiry     TERMINAL  UNDERSTAND -> UNDERSTAND -> UNDERSTAND

    Trois compréhensions/consécutions consécutives en fin d'historique : dialogue clos.

OK  inquiry     TERMINAL  CLAIM -> QUESTION -> UNDERSTAND -> UNDERSTAND

    Deux compréhensions consécutives en position 4 : condition de convergence atteinte.

OK  inquiry     TERMINAL  CLAIM -> SUPPORT -> CLAIM -> SUPPORT -> CLAIM -> SUPPORT

    Boucle de motif (claim, support) répétée trois fois : le dialogue tourne en rond, on coupe.

... (5 outputs left out)

    Deux retraits consécutifs : plus rien à persuader.

OK  persuasion  en cours  CLAIM -> CHALLENGE -> ARGUE

    Persuasion en cours, pas de concession ni retrait final.

7/7 historiques rendent le verdict attendu : True

### Lecture : une entrée à double fond

L'historique `CHALLENGE -> RETRACT -> RETRACT` est consigné **TERMINAL** par la
fixture — et pourtant la table de transitions interdit d'y arriver : seul
`CHALLENGE` mène à `RETRACT`, et `RETRACT` ne mène pas à `RETRACT`. La condition
de terminaison `_term_double_retract` vit donc sur un état qu'aucun dialogue
légal ne peut produire. Ce n'est pas une erreur de la fixture : elle teste
l'état, pas la légalité du chemin — et le croisement des deux tables met au jour
une propriété du moteur que la section finale mesure en détail.


## La trace, unité d'observation

L'objet que manipule le validateur n'est pas le dialogue en train de se dérouler
mais sa *trace* : la liste des coups joués, dans l'ordre. Chaque coup est un
`DialogueMove` — un locuteur, un acte, un contenu libre et une cible optionnelle.
La machine à états ne lit que l'acte : le contenu sert à l'humain qui relit la
trace, pas au protocole.

`ValidateTrace(moves)` rend quatre choses : la validité transition par transition
(`Valid`), l'index du premier coup illégal s'il y en a un (`FirstInvalid`), la
terminaison (`Terminated`) et la condition de terminaison satisfaite (`Reason`).
C'est l'instrument unique de ce carnet : chaque expérience qui suit se lit dans
ces quatre champs.


In [8]:
var traceInq = new List<DialogueMove>
{
    new("A", SpeechAct.QUESTION, "Le seuil de 80 % est-il atteint ?"),
    new("B", SpeechAct.CLAIM, "Oui, la mesure du jour donne 82 %"),
    new("A", SpeechAct.SUPPORT, "J'ajoute : hier on etait deja a 81 %"),
    new("B", SpeechAct.UNDERSTAND, "Nos deux mesures se corroborent"),
    new("A", SpeechAct.UNDERSTAND, "Compris, nous sommes d'accord sur le fait"),
};
var res = inq.ValidateTrace(traceInq);
Show("Trace :", string.Join(" -> ", traceInq.Select(m => m.Act)));
Show($"valid : {res.Valid} | first_invalid : {res.FirstInvalid?.ToString() ?? "None"}");
Show($"terminated : {res.Terminated} | reason : {res.Reason}");

Trace :QUESTION -> CLAIM -> SUPPORT -> UNDERSTAND -> UNDERSTAND

valid : True | first_invalid : None

terminated : True | reason : _term_converged

### Lecture : conclure sans gagner

Les cinq coups sont légitimes (`valid: True`, `first_invalid: None`) et le
dialogue est terminé par `_term_converged` : deux `UNDERSTAND` consécutifs.
Personne n'a cédé, personne n'a gagné : les deux parties déclarent avoir compris
la même chose. C'est la signature de l'inquiry — l'accord sur les faits, pas la
victoire d'une thèse. Noter que la condition ne regarde que la fin de la trace :
les deux derniers coups suffisent, le reste de l'historique est indifférent à
cette terminaison-là.


In [9]:
var paires = new (string sp, SpeechAct act)[] { ("A", SpeechAct.QUESTION), ("B", SpeechAct.CLAIM) };
var traceLoop = Enumerable.Range(0, 3).SelectMany(_ => paires.Select(p => new DialogueMove(p.sp, p.act))).ToList();
var resLoop = inq.ValidateTrace(traceLoop);
Show("Trace :", string.Join(" -> ", traceLoop.Select(m => m.Act)));
Show($"valid : {resLoop.Valid} | terminated : {resLoop.Terminated}");
Show($"reason : {resLoop.Reason}");

Trace :QUESTION -> CLAIM -> QUESTION -> CLAIM -> QUESTION -> CLAIM

valid : True | terminated : True

reason : _term_loop

### Lecture : légal et mort à la fois

`valid: True` **et** `terminated: True` : chaque transition est permise, et
pourtant le protocole déclare le dialogue terminé (`_term_loop`). La même paire
d'actes `(QUESTION, CLAIM)` répétée trois fois est irréprochable coup par coup et
vide dans son ensemble. C'est la limite de la validité transitionnelle — elle ne
voit que les enchaînements locaux — et la raison pour laquelle la terminaison est
une couche *séparée* de la table : détecter qu'un dialogue ne progresse plus
demande de regarder la forme globale de la trace, pas seulement ses arêtes.


In [10]:
var motif3 = new (string sp, SpeechAct act)[]
{
    ("A", SpeechAct.QUESTION), ("B", SpeechAct.CLAIM), ("A", SpeechAct.SUPPORT)
};
var traceP3 = Enumerable.Range(0, 3).SelectMany(_ => motif3.Select(p => new DialogueMove(p.sp, p.act))).ToList();
Show("Motif de periode 3 :", string.Join(", ", traceP3.Take(3).Select(m => m.Act)) + " (repete trois fois)");
var resP3 = inq.ValidateTrace(traceP3);
Show($"valid : {resP3.Valid} | terminated : {resP3.Terminated} | reason : {resP3.Reason}");

Motif de periode 3 :QUESTION, CLAIM, SUPPORT (repete trois fois)

valid : True | terminated : False | reason : 

### Lecture : la limite mesurée du détecteur

Le motif `(QUESTION, CLAIM, SUPPORT)` répété trois fois passe : `terminated:
False`, aucune raison de terminaison. Neuf coups qui tournent en rond traversent
le filtre, alors que six coups en période 2 étaient coupés juste avant. La cause
est dans la définition du détecteur : il compare les *paires* adjacentes des six
derniers coups — `recent[0:2] == recent[2:4] == recent[4:6]` — et une période 3
ne fait jamais coïncider ces paires. C'est la divergence 3 du port, mesurée ici
sur une trace exécutée ; l'exercice 1 fait écrire le détecteur de période 3 qui
manque.


## Persuasion : le même formalisme, une autre nature

Le passage de l'inquiry à la persuasion ne change ni le formalisme ni
l'instrument : mêmes neuf actes, même `ValidateTrace`. Ce qui change est la
*table* — et à travers elle, la nature de l'échange. L'inquiry est coopératif :
deux parties construisent un accord sur les faits, et personne ne « gagne ». La
persuasion est adversariale : deux thèses s'affrontent, et le dialogue se conclut
par une capitulation — `CONCEDE` — ou par épuisement.

Où se lit l'adversarialité ? Dans les transitions elles-mêmes. `REFUTE` est
vivant en persuasion (on peut attaquer une thèse, et l'attaque appelle une
riposte) ; il est quasi muet en inquiry (on demande, on clarifie, on ne détruit
pas). `RETRACT` n'existe qu'en persuasion : dans une enquête on ne retire pas sa
parole, on la précise. La table de transitions est donc une théorie du dialogue
*encodée en machine* — les expériences qui suivent la lisent coup par coup.


In [11]:
var tracePer = new List<DialogueMove>
{
    new("A", SpeechAct.CLAIM, "La dessalinisation est la meilleure option pour la ville"),
    new("B", SpeechAct.CHALLENGE, "Pourquoi celle-la plutot que le recyclage ?"),
    new("A", SpeechAct.ARGUE, "Cout par m3 le plus bas a l'echelle de nos besoins"),
    new("B", SpeechAct.CONCEDE, "Soit : a cette echelle, j'accepte l'argument"),
};
var resPer = per.ValidateTrace(tracePer);
Show("Trace :", string.Join(" -> ", tracePer.Select(m => m.Act)));
Show($"valid : {resPer.Valid} | terminated : {resPer.Terminated}");
Show($"reason : {resPer.Reason}");

Trace :CLAIM -> CHALLENGE -> ARGUE -> CONCEDE

valid : True | terminated : True

reason : _term_concede

### Lecture : la victoire codée

`reason: _term_concede` — le dernier coup est une capitulation, et cela suffit.
Comparée à la trace d'inquiry (conclure par `UNDERSTAND`), la différence tient en
un acte : le même formalisme, un `CONCEDE` à la place d'un `UNDERSTAND`, et le
sens de la fin change — victoire d'une thèse contre accord sur un fait. Aucune
comptabilité de score n'est nécessaire : la condition de terminaison *est* le
verdict.


In [12]:
var menentARetract = Enum.GetValues(typeof(SpeechAct)).Cast<SpeechAct>()
    .Where(a => per.AllowedTransitions.TryGetValue(a, out var s) && s.Contains(SpeechAct.RETRACT))
    .Select(a => a.ToString()).ToList();
Show("Actes menant a RETRACT (persuasion) :", string.Join(", ", menentARetract));
Show("Successeurs de RETRACT              :", string.Join(", ", per.AllowedTransitions[SpeechAct.RETRACT]));

var impasse = new List<DialogueMove>
{
    new("A", SpeechAct.CHALLENGE),
    new("B", SpeechAct.RETRACT, "Je retire ma these"),
    new("A", SpeechAct.RETRACT, "Moi aussi"),
};
var resImp = per.ValidateTrace(impasse);
Console.WriteLine();
Show("Trace CHALLENGE -> RETRACT -> RETRACT :");
Show($"valid : {resImp.Valid} | first_invalid : {resImp.FirstInvalid}");
Show($"is_terminal_state : {per.IsTerminalState(impasse)}");

Actes menant a RETRACT (persuasion) :CHALLENGE

Successeurs de RETRACT              :CLAIM, QUESTION

Trace CHALLENGE -> RETRACT -> RETRACT :

valid : False | first_invalid : 2

is_terminal_state : True

### Lecture : une condition de terminaison morte

La trace `CHALLENGE -> RETRACT -> RETRACT` est **invalide à l'index 2**
(`RETRACT` n'admet que `CLAIM` et `QUESTION`), et pourtant `IsTerminalState`
rend `True` : la condition `_term_double_retract` est satisfaite par une trace
qu'aucun dialogue légal ne peut produire. En effet, seul `CHALLENGE` mène à
`RETRACT`, et `RETRACT` ne mène pas à `RETRACT` : deux retraits consécutifs
sont un état interdit par la table. La condition vit dans le code et reste morte
dans le dialogue — c'est la divergence 5 du port, découverte en écrivant cette
démonstration. Leçon de spécification : une machine à états se vérifie aussi du
côté de ses conditions de terminaison — chacune doit correspondre à un état
*atteignable* par les transitions permises.


In [13]:
var traceBad = new List<DialogueMove>
{
    new("A", SpeechAct.REFUTE, "Votre mesure est biaisee par la saison"),
    new("B", SpeechAct.CONCEDE, "Vous avez raison"),
};
var resBad = inq.ValidateTrace(traceBad);
Show("Trace REFUTE -> CONCEDE en inquiry :");
Show($"valid : {resBad.Valid} | first_invalid : {resBad.FirstInvalid}");
Show("REFUTE n'admet que :", string.Join(", ", inq.GetAllowedResponses(SpeechAct.REFUTE)));
Show($"Meme trace en persuasion : valid = {per.ValidateTrace(traceBad).Valid} (REFUTE -> CONCEDE y est permise)");

Trace REFUTE -> CONCEDE en inquiry :

valid : False | first_invalid : 1

REFUTE n'admet que :ARGUE, QUESTION

Meme trace en persuasion : valid = True (REFUTE -> CONCEDE y est permise)

### Lecture : illégal relativement à quoi ?

`REFUTE -> CONCEDE` est rejeté à l'index 1 en inquiry : une réfutation appelle
des raisons (`ARGUE`) ou une précision (`QUESTION`), jamais une capitulation. La
même trace, exécutée sur la table de persuasion, est **légale** : là, `REFUTE`
admet explicitement `CONCEDE` — détruire un argument peut faire céder
l'adversaire. Un coup n'est donc pas illégal dans l'absolu : il est illégal
*relativement à un protocole*. C'est tout ce que « protocoliser » achète : la
contestation « tu n'avais pas le droit de conclure ici » reçoit une réponse
mécanique, tranchée par la table et localisée à l'index près.


In [14]:
// Tableau croise : 9 actes x 2 protocoles (memorandum de la cellule du jumeau Python).
var headers = new[] { "acte", "inquiry", "persuasion" };
var rows = Enum.GetValues(typeof(SpeechAct)).Cast<SpeechAct>()
    .Select(act => new string[]
    {
        act.ToString(),
        inq.GetAllowedResponses(act).Count > 0 ? string.Join(",", inq.GetAllowedResponses(act)) : "(muet)",
        per.GetAllowedResponses(act).Count > 0 ? string.Join(",", per.GetAllowedResponses(act)) : "(muet)",
    }).ToList();
Show(Table(headers, rows));

acte       | inquiry                               | persuasion                        
-----------+---------------------------------------+-----------------------------------
CLAIM      | SUPPORT,CHALLENGE,QUESTION,UNDERSTAND | CHALLENGE,CONCEDE,QUESTION,SUPPORT
QUESTION   | CLAIM,ARGUE,QUESTION,SUPPORT          | CLAIM,ARGUE,SUPPORT               
CHALLENGE  | ARGUE,SUPPORT,QUESTION                | ARGUE,RETRACT,SUPPORT             
ARGUE      | CHALLENGE,UNDERSTAND,QUESTION,SUPPORT | CHALLENGE,CONCEDE,REFUTE,SUPPORT  
CONCEDE    | QUESTION,UNDERSTAND                   | CLAIM,QUESTION                    
RETRACT    | (muet)                                | CLAIM,QUESTION                    
SUPPORT    | QUESTION,UNDERSTAND,CHALLENGE         | UNDERSTAND,CHALLENGE,QUESTION     
REFUTE     | ARGUE,QUESTION                        | ARGUE,CONCEDE,CHALLENGE           
UNDERSTAND | QUESTION,UNDERSTAND,CLAIM             | CLAIM,QUESTION                    


### Lecture croisée : la nature se lit dans la table

Quatre différences structurelles sautent du tableau :

1. **`RETRACT` est muet en inquiry, vivant en persuasion** — dans une enquête on
   précise sa parole, on ne la retire pas ; dans un affrontement, se rétracter
   est un coup tactique.
2. **`UNDERSTAND` boucle sur lui-même en inquiry** (`UNDERSTAND` est dans ses
   propres successeurs — c'est le mécanisme exact de `_term_converged`), **pas
   en persuasion** où il ne mène qu'à `CLAIM` ou `QUESTION` : comprendre est une
   fin en enquête, un simple tour de passe en dispute.
3. **`SUPPORT` ouvre sur `UNDERSTAND` en persuasion** mais sur `QUESTION` en
   inquiry : soutenir y appelle une réaction immédiate, ici une clarification.
4. **`CONCEDE` est la cible de trois actes en persuasion, d'aucun en inquiry** —
   la porte de sortie existe à chaque étage du monde adversarial, et n'existe
   nulle part dans le monde coopératif.

La nature d'un dialogue n'est donc pas un attribut déclaratif : c'est la
composition de sa table — et les colonnes ci-dessus en sont la radiographie.


## Le dialogue de grounding : le vrai Tweety, nativement

La machine à neuf actes juge la *légalité locale* d'un coup : la table de
transitions dit si un enchaînement est permis, la condition de terminaison dit
si le dialogue est fini — elle ne dit jamais qui *a raison*. Le pendant de
production répond à cette seconde question par un autre organe :
`DialogueHandler.execute_dialogue` (porté dans
`argumentation_lib/_dialogue_handler.py`), qui exécute un dialogue de *grounding*
proponent/opponent — fusion des deux vues du cadre de Dung, extension *fondée*
calculée par `SimpleGroundedReasoner`, verdict `accepted` / `rejected` sur la
thèse débattue. Là où la table demandait « ce coup est-il jouable ? », le
grounding demande « cette thèse survit-elle au cadre fusionné ? » — et le verdict
**est** la sémantique, pas une condition de terminaison ad hoc.

**C'est ici que le jumeau C# diverge du jumeau Python — en mieux.** Le pont
Python passe par `jpype` et un JVM provisionné par le carnet `00-Setup`. Ce
jumeau charge le **même organe, nativement** : les classes
`org.tweetyproject.arg.dung.syntax.DungTheory` / `Argument` / `Attack` et
`org.tweetyproject.arg.dung.reasoner.SimpleGroundedReasoner` sont celles du
vrai Tweety, compilées en C# par IKVM et embarquées dans le shade
`org.tweetyproject.tweety-dung.dll` du dépôt (chargé au setup, présence
vérifiée par réflexion). Reprenons le débat de la trace de persuasion — la
dessalinisation (thèse de A) contre le recyclage (alternative de B) — traduit en
cadre de Dung, puis une variante où B ajoute un argument sans réponse.


In [15]:
// --- Dialogue de grounding sur le VRAI Tweety (shade IKVM charge au setup) ---
// Portage de DialogueHandler.execute_dialogue en C# natif : memes classes
// DungTheory / Argument / Attack / SimpleGroundedReasoner — pas de JVM, pas
// de pont jpype.

class DialogueResult
{
    public string Topic = "";
    public string Outcome = "";
    public List<string> GroundedExtension = new();
    public List<Dictionary<string, object>> DialogueTrace = new();
    public List<string> MergedArguments = new();
    public List<List<string>> MergedAttacks = new();
    public int Rounds => DialogueTrace.Count;
    public int GroundedSize => GroundedExtension.Count;
}

static DialogueResult ExecuteDialogue(
    List<string> proponentArgs, List<List<string>> proponentAttacks,
    List<string> opponentArgs, List<List<string>> opponentAttacks,
    string topic, int maxRounds = 10)
{
    // Cadre fusionne (portage de la fusion des vues du handler).
    var allArgs = proponentArgs.Concat(opponentArgs).Distinct().ToList();
    var allAttacks = new List<List<string>>();
    var seen = new HashSet<string>();
    foreach (var atk in proponentAttacks.Concat(opponentAttacks))
        if (seen.Add(atk[0] + "->" + atk[1])) allAttacks.Add(atk);

    var theory = new org.tweetyproject.arg.dung.syntax.DungTheory();
    var argMap = allArgs.ToDictionary(n => n,
        n => new org.tweetyproject.arg.dung.syntax.Argument(n));
    foreach (var a in argMap.Values) theory.add(a);
    foreach (var atk in allAttacks)
        if (argMap.TryGetValue(atk[0], out var src) && argMap.TryGetValue(atk[1], out var tgt))
            theory.add(new org.tweetyproject.arg.dung.syntax.Attack(src, tgt));

    // Extension fondee (SimpleGroundedReasoner : meme organe que le pont jpype).
    var reasoner = new org.tweetyproject.arg.dung.reasoner.SimpleGroundedReasoner();
    var grounded = (org.tweetyproject.arg.dung.semantics.Extension[])
        reasoner.getModels(theory).toArray(
            new org.tweetyproject.arg.dung.semantics.Extension[0]);
    var groundedArgs = new List<string>();
    if (grounded.Length > 0)
        foreach (var a in grounded[0]) groundedArgs.Add(a.ToString());
    groundedArgs.Sort();

    // Trace du dialogue (portage de la simulation proponent/opponent).
    var trace = new List<Dictionary<string, object>>
    {
        new() { ["round"] = 1, ["speaker"] = "proponent", ["action"] = "assert", ["argument"] = topic }
    };
    int roundNum = 2;
    foreach (var atk in opponentAttacks)
    {
        if (atk[1] == topic || proponentAttacks.Any(pa => pa[0] == atk[1]))
        {
            trace.Add(new()
            {
                ["round"] = roundNum, ["speaker"] = "opponent",
                ["action"] = "attack", ["argument"] = atk[0], ["target"] = atk[1]
            });
            if (++roundNum > maxRounds) break;
        }
    }
    foreach (var atk in proponentAttacks)
        foreach (var oppAtk in opponentAttacks)
            if (atk[1] == oppAtk[0])
            {
                trace.Add(new()
                {
                    ["round"] = roundNum, ["speaker"] = "proponent",
                    ["action"] = "defend", ["argument"] = atk[0], ["target"] = atk[1]
                });
                if (++roundNum > maxRounds) break;
            }

    return new DialogueResult
    {
        Topic = topic,
        Outcome = groundedArgs.Contains(topic) ? "accepted" : "rejected",
        GroundedExtension = groundedArgs,
        DialogueTrace = trace,
        MergedArguments = allArgs.OrderBy(x => x).ToList(),
        MergedAttacks = allAttacks,
    };
}

// Meme debat que la trace de persuasion : la dessalinisation (these de A) contre
// le recyclage (alternative de B), traduit en cadre de Dung.
var resDessal = ExecuteDialogue(
    proponentArgs: new() { "dessalinisation", "cout_m3" },
    proponentAttacks: new() { new() { "cout_m3", "recyclage" } },
    opponentArgs: new() { "recyclage" },
    opponentAttacks: new() { new() { "recyclage", "dessalinisation" } },
    topic: "dessalinisation");
Show("Cadre fusionne :", string.Join(", ", resDessal.MergedArguments));
Show("Attaques       :", string.Join(" ; ", resDessal.MergedAttacks.Select(a => $"{a[0]} -> {a[1]}")));
Show("Extension fondee :", string.Join(", ", resDessal.GroundedExtension));
Show("Verdict sur la these :", resDessal.Outcome);
Show($"Trace ({resDessal.Rounds} rounds) :");
foreach (var step in resDessal.DialogueTrace)
{
    var cible = step.TryGetValue("target", out var t) ? " -> " + t : "";
    Show($"  round {step["round"]} | {step["speaker"]} | {step["action"]} | {step["argument"]}{cible}");
}

// Variante : B ajoute un argument d'empreinte carbone qui attaque la these sans
// rencontrer de defense. Protocole identique, un argument de plus -- verdict ?
var resCarbone = ExecuteDialogue(
    proponentArgs: new() { "dessalinisation", "cout_m3" },
    proponentAttacks: new() { new() { "cout_m3", "recyclage" } },
    opponentArgs: new() { "recyclage", "empreinte_carbone" },
    opponentAttacks: new()
    {
        new() { "recyclage", "dessalinisation" },
        new() { "empreinte_carbone", "dessalinisation" },
    },
    topic: "dessalinisation");
Console.WriteLine();
Show("Variante carbone -- extension fondee :", string.Join(", ", resCarbone.GroundedExtension));
Show("Variante carbone -- verdict :", resCarbone.Outcome);

Cadre fusionne :cout_m3, dessalinisation, recyclage

Attaques       :cout_m3 -> recyclage ; recyclage -> dessalinisation

Extension fondee :cout_m3, dessalinisation

Verdict sur la these :accepted

Trace (3 rounds) :

  round 1 | proponent | assert | dessalinisation

  round 2 | opponent | attack | recyclage -> dessalinisation

  round 3 | proponent | defend | cout_m3 -> recyclage

Variante carbone -- extension fondee :cout_m3, empreinte_carbone

Variante carbone -- verdict :rejected

### Lecture : légalité locale contre sort global

Les deux organes ne répondent pas à la même question, et l'écart se lit sur le
même débat. Dans la trace de persuasion, `CONCEDE` terminait le dialogue : B
capitule, la condition de terminaison *est* le verdict — mais rien n'a vérifié
que l'argument de A tenait. Ici le verdict `accepted` ne vient d'aucune
capitulation : `cout_m3` n'est attaqué par personne, il bat `recyclage`, donc il
défend `dessalinisation` — la thèse est dans l'extension fondée, la structure du
cadre tranche. Dans la variante carbone, B ne concède rien : son
`empreinte_carbone`, non contredit, suffit à sortir la thèse de l'extension — le
verdict bascule à `rejected` alors que la table de transitions n'a rien de plus
à dire. Les deux divergences mesurées plus haut (période 3 invisible au
détecteur, `_term_double_retract` mort) sont des défauts de la question
*légalité* ; c'est pourquoi le tronc a retiré cette machine et ne conserve en
production que la question *sort global* — calculée ici sur le vrai Tweety,
nativement.


### Les neuf actes, côté production : un canal déclaré sans producteur

Le vocabulaire Walton-Krabbe de ce carnet a bien eu un répondant de production —
mais **dans le tronc EPITA, pas dans ce dépôt**. Dans
`argumentation_analysis/agents/core/informal/dung_arbitration_stage.py` du dépôt
[`2025-Epita-Intelligence-Symbolique`](https://github.com/jsboigeEpita/2025-Epita-Intelligence-Symbolique),
la constante `WALTON_KRABBE_ATTACKING_ACTS` séparait les actes attaquants
(`REFUTE`, `CHALLENGE` — ceux qui deviennent des attaques Dung) des actes
coopératifs (`CONCEDE`, `SUPPORT`, `UNDERSTAND`), et
`walton_krabbe_conflict_policy` transformait les relations déclarées entre
candidats en attaques.

Deux réserves mesurées, avant de lire ce rapprochement comme un « usage de
production ».

**Ce canal n'est pas porté ici.** `dung_arbitration_stage.py` ne figure pas dans
le `NOTICE-EPITA` de `argumentation_lib/` : aucune copie verbatim n'en a été
vendue dans CoursIA. Le dialogue de grounding exécuté plus haut, lui, **est**
porté (`_dialogue_handler.py`) — c'est l'écart entre citer un organe et en
exécuter un.

**Et il n'existe plus en amont.** Le 2026-10-05, le commit `166a5e7c` l'a retiré
avec un motif qui vaut leçon : « retire the Walton-Krabbe relations channel,
whose producer never existed ». Le canal lisait
`context["walton_krabbe_relations"]` alors que **personne n'écrivait jamais cette
clé** ; `DebateAgent` produit des scores de qualité, pas des actes
`REFUTE`/`CHALLENGE`.

C'est l'anti-modèle exact de ce que ce carnet enseigne : une politique qui
transforme des relations que nul ne fournit reste cohérente tant qu'on ne demande
pas **qui les produit**. Les neuf actes gardent ici leur valeur de
**spécification exécutable** — table de transitions, condition de terminaison,
dialogue de grounding réellement exécuté — et non celle d'une politique branchée
sur un flux inexistant.


## Jonction avec les assets voisins : le schéma d'un argument du dialogue

Le protocole échange des *actes* ; l'argument lui-même a une structure. Deux
assets voisins de la série la portent : le classifieur de schémas de Walton
(`argumentation_schemes.py`, carnet `01b`) reconnaît, sur son texte, le schéma
d'un argument — appel à l'expert, exemple, cause… — et la base de connaissances
(`knowledge_base.py`, carnet `04b`) structure prémisses, conclusion et schéma
dans un `FormalArgument`. Le carnet de référence joint les trois niveaux : un
argument du dialogue dont le champ `scheme` est peuplé par le classifieur.

Dans ce jumeau C#, le classifieur Walton **reste du côté Python** (carnet `01b`)
— aucune dépendance C# n'a été portée pour cet asset : la cellule suivante pose
donc le schéma explicitement, et le déclare.


In [16]:
// Un FormalArgument dont le schema est pose explicitement. (Difference declaree :
// le classifieur Walton — asset 01b — reste du cote Python ; pas de portage du
// classifieur dans ce jumeau.)
class PropositionCSharp
{
    public string Content { get; }
    public double Confidence { get; }
    public PropositionCSharp(string content, double confidence = 1.0)
    { Content = content; Confidence = confidence; }
    public override string ToString() => Content;
}

class FormalArgumentCSharp
{
    public List<PropositionCSharp> Premises { get; }
    public PropositionCSharp Conclusion { get; }
    public string? Scheme { get; }
    public string Id { get; }
    public FormalArgumentCSharp(List<PropositionCSharp> premises,
        PropositionCSharp conclusion, string? scheme = null)
    { Premises = premises; Conclusion = conclusion; Scheme = scheme; Id = Guid.NewGuid().ToString(); }
    public override string ToString() =>
        $"[{string.Join(", ", Premises.Select(p => (object)p))}] -> {Conclusion}";
}

var argument = new FormalArgumentCSharp(
    premises: new() { new PropositionCSharp("Le chercheur est specialiste du domaine", 0.9) },
    conclusion: new PropositionCSharp("La methode est fiable", 0.8),
    scheme: "expert_opinion");
Show($"Argument   : {argument}");
Show($"Schema     : {argument.Scheme} (pose manuellement — classifieur du jumeau Python 01b)");
Show($"Identifiant: {argument.Id[..8]}...");

Argument   : [Le chercheur est specialiste du domaine] -> La methode est fiable

Schema     : expert_opinion (pose manuellement — classifieur du jumeau Python 01b)

Identifiant: 4ebd5c89...


(20,45): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(17,18): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



### Lecture : trois niveaux, une seule chaîne

Le classifieur rend un `ArgumentationScheme` (`expert_opinion`, force a priori),
la base range l'argument (`[premisses] -> conclusion`) et le protocole, lui,
dira quand cet argument peut être *posé* (`CLAIM`), *soutenu* (`SUPPORT`) ou
*réfuté* (`REFUTE`). Le champ `scheme` est la clé de jonction : c'est elle qu'une
démonstration automatique lirait pour relier le tour de parole à la théorie de
l'argumentation qu'il instancie.

## Propriétés mesurées du moteur

Le moteur est volontairement **documenté plutôt que corrigé** : ses angles morts
sont mesurables, et chacun enseigne ce qu'un protocole bien spécifié doit
vérifier.

1. **Six types déclarés, deux machines.** Le registre Walton–Krabbe énumère six
   types de dialogue ; seuls `INQUIRY` et `PERSUASION` ont un protocole
   implémenté. Une typologie nomme plus qu'elle ne construit — l'écart est en
   soi instructif.
2. **La boucle de période 3 n'est pas vue.** Le détecteur de boucle compare les
   paires d'actes adjacentes sur les six derniers coups : un motif de période 2
   répété est coupé, un motif de période 3 traverse le filtre (démonstration plus
   haut, exercice 1).
3. **Limites de longueur asymétriques.** 25 coups pour l'inquiry, 30 pour la
   persuasion, sans justification documentée — portées telles quelles.
4. **La condition `_term_double_retract` est inatteignable.** Elle teste deux
   `RETRACT` consécutifs — un état que la table de transitions interdit : seul
   `CHALLENGE` mène à `RETRACT`, et `RETRACT` ne mène pas à `RETRACT`. La
   fixture partagée le met en évidence sans le nommer : elle consigne
   l'historique `CHALLENGE -> RETRACT -> RETRACT` comme TERMINAL (la condition
   tire sur l'état), alors qu'aucun dialogue légal ne peut l'atteindre (la table
   interdit le chemin).

Documenter plutôt que corriger est ici une décision d'ingénierie : l'essence d'un
moteur de protocole, ce sont aussi ses angles morts. Ils enseignent ce qu'un
protocole bien spécifié doit vérifier — ici, que chaque condition de terminaison
corresponde à un état réellement atteignable par les transitions permises.


## Vérifiez votre compréhension

Trois exercices, un par acquisition. Les stubs s'exécutent sans erreur (règle
C.1) et la méthode de chaque exercice est déjà démontrée sur un exemple dans le
carnet :

- **Exercice 1** (détecteur de période 3) → la cellule « motif 3 » l'a montré en
  sortie : `valid : True` mais `terminated : False` avec `reason : ` null — le
  motif de période 3 est légal et ne meurt d'aucune des conditions de
  terminaison ; l'indice `motif[0:3] == motif[3:6] == motif[6:9]` transpose le
  geste de la démo.
- **Exercice 2** (persuasion de 31 coups) → les cellules de terminaison ont
  montré les deux morts à éviter : `reason : _term_concede` (sur `CLAIM ->
  CHALLENGE -> ARGUE -> CONCEDE`) et `_term_double_retract` (sur `CHALLENGE ->
  RETRACT -> RETRACT`) ; la boucle de quatre actes se vérifie arête par arête
  avec `IsValidMove`, comme la fixture l'a fait sur ses 9 transitions.
- **Exercice 3** (qui meurt, et comment) → la fixture a énuméré les causes
  possibles (`_term_converged`, `_term_loop`, `_term_concede`, `_term_length`) ;
  la simulation seedée mesure leur distribution, inquiry contre persuasion — on
  la lance pour savoir, pas pour confirmer.


## Exercice 1 — le détecteur de période 3

Le détecteur de boucle du protocole ne voit que les répétitions de *paires*
d'actes (période 2). Écrire `DetectPeriod3Loop(history)` qui rend `true` si et
seulement si les neuf derniers coups forment trois répétitions d'un même triplet
d'actes, et `false` sinon.

Témoins attendus :

- la trace `traceP3` de la démonstration ci-dessus doit rendre `true` ;
- la trace `traceLoop` (période 2 : `(QUESTION, CLAIM)` répété) doit rendre
  `false` ;
- une trace de moins de neuf coups doit rendre `false`.

Indications : extraire les neuf derniers actes dans une liste `motif`, puis
comparer le triplet de tête aux deux suivants — `motif[0:3] == motif[3:6] ==
motif[6:9]` (en C# : comparer les triplets `(motif[i], motif[i+1], motif[i+2])`).


In [17]:
static bool DetectPeriod3Loop(List<DialogueMove> history)
{
    // TODO etudiant : rendre true ssi les 9 derniers coups forment un motif de
    // periode 3 repete trois fois, false sinon.
    // Etape 1 : si moins de 9 coups, rendre false.
    // Etape 2 : extraire les 9 derniers actes dans une liste motif.
    // Etape 3 : rendre motif[0:3] == motif[3:6] == motif[6:9].
    return false; // TODO etudiant
}

Show("Exercice 1 a completer");

Exercice 1 a completer

## Exercice 2 — une persuasion qui ne meurt jamais (presque)

Construire une trace **légale** de persuasion de 31 coups qui atteint la
condition `_term_length` (plus de 30 coups) sans jamais satisfaire ni
`_term_concede` ni `_term_double_retract` : autrement dit, un dialogue où
personne ne cède et personne ne retire deux fois de suite, jusqu'à épuisement du
temps imparti.

La fonction `TracePersuasionLongue()` doit renvoyer la liste des
`DialogueMove`. Le témoin est `ValidateTrace` : `Valid` doit valoir `true` et
`Reason` doit valoir `_term_length`.

Indication : il existe un cycle de quatre actes légal en persuasion, par exemple
`CHALLENGE -> SUPPORT -> QUESTION -> CLAIM -> CHALLENGE -> ...` — vérifier chaque
arête avec `IsValidMove` avant d'écrire la boucle.


In [18]:
static List<DialogueMove> TracePersuasionLongue()
{
    // TODO etudiant : construire 31 DialogueMove formant une trace legale de
    // persuasion, terminee uniquement par la limite de longueur.
    // Etape 1 : choisir un cycle d'actes dont chaque transition est valide.
    // Etape 2 : generer 31 coups en alternant les locuteurs A et B.
    // Etape 3 : verifier avec per.ValidateTrace (Valid: true, Reason: _term_length).
    return new List<DialogueMove>(); // TODO etudiant
}

Show("Exercice 2 a completer");

Exercice 2 a completer

## Exercice 3 — qui meurt, et comment ?

Simuler 200 dialogues par protocole et compter les causes de mort. Deux
locuteurs A et B alternent ; chaque coup est tiré au hasard (générateur seedé)
parmi les réponses permises au dernier acte ; le dialogue démarre sur `QUESTION`
(inquiry) ou `CLAIM` (persuasion) et s'arrête dès que `IsTerminalState` est
vrai. La fonction `SimulerDialogues(protocole, n, seed)` doit renvoyer un
dictionnaire `{raison_de_terminaison: compte}`.

Questons auxquelles la distribution répondra : l'inquiry meurt-elle plutôt par
compréhension mutuelle ou par longueur ? La persuasion cède-t-elle
(`_term_concede`) plus souvent qu'elle ne s'épuise (`_term_length`) ? La réponse
n'est pas intuitive — c'est le propre d'une expérience : on la lance pour
savoir, pas pour confirmer.


In [19]:
static Dictionary<string, int> SimulerDialogues(DialogueProtocol protocole, int n = 200, int seed = 0)
{
    // TODO etudiant : simuler n dialogues et compter les raisons de terminaison.
    // Etape 1 : pour chaque dialogue i, tirer les coups avec new Random(seed + i).
    // Etape 2 : demarrer sur QUESTION (inquiry) ou CLAIM (persuasion).
    // Etape 3 : chaque coup suivant est un tirage parmi GetAllowedResponses(acte).
    // Etape 4 : arreter des que IsTerminalState(trace) est vrai ; consigner
    //           la raison (TerminationReason) dans un dictionnaire {raison: compte}.
    return new Dictionary<string, int>(); // TODO etudiant
}

Show("Exercice 3 a completer");

Exercice 3 a completer

## À retenir

Ce carnet a lu deux protocoles de dialogue comme ce qu'ils sont : des machines à
états sur des actes de langage. Trois choses à retenir :

1. **La table de transitions EST la théorie.** Coopératif ou adversarial ne se
   décrivent pas en adjectifs : `REFUTE` vivant ou muet, `RETRACT` présent ou
   absent, `CONCEDE` cible de trois actes ou d'aucun — la nature du dialogue se
   lit dans la table, coup par coup. Mesuré : `Transitions inquiry : 25` contre
   `26` en persuasion ; `REFUTE -> CONCEDE` est `valid : False` en inquiry
   (`first_invalid : 1`, REFUTE n'admet que ARGUE et QUESTION) mais
   `valid = True` en persuasion ; CONCEDE est ciblée par `CLAIM, ARGUE, REFUTE`
   en persuasion et par **aucun** acte en inquiry.
2. **La terminaison se spécifie, sinon elle se décrète.** Le protocole coupe les
   dialogues qui tournent en rond et énumère ses morts ; la divergence 5 enseigne
   la réciproque — une condition de terminaison écrite pour un état que la table
   interdit est du code mort, qu'on ne découvre qu'en cherchant à le déclencher
   — mesuré : `CHALLENGE -> RETRACT -> RETRACT` est `valid : False` (le second
   RETRACT est illégal) et pourtant `is_terminal_state : True`.
3. **Le formalisme est léger, les conséquences ne le sont pas.** Neuf actes et
   une vingtaine de transitions suffisent à rendre une contestation tranchable,
   une terminaison garantie et une expérience reproductible — la trace
   `QUESTION -> CLAIM -> SUPPORT -> UNDERSTAND -> UNDERSTAND` conclut sans gain
   (`terminated : True`, `reason : _term_converged`).

Et, propre à ce jumeau : **le grounding tourne sur le vrai Tweety, nativement.**
Même `DungTheory`, mêmes `Argument`/`Attack`, même `SimpleGroundedReasoner` que
le pont `jpype` du jumeau Python — sans JVM : la dessalinisation est `accepted`
(`cout_m3` non attaqué défend la thèse), la variante carbone est `rejected`
(`empreinte_carbone` non contredit la fait sortir de l'extension fondée).


## Pour aller plus loin

- [Index de la sous-série ArgumentAnalysis](README.md) — la place des protocoles
  de dialogue dans le parcours.
- [Index de la série SymbolicAI](../README.md) — le fil complet, du raisonnement
  formel aux systèmes multi-agents.

Le carnet [03d (ADF)](Argumentation-03d-ADF-Semantiques-Python.ipynb)
généralise l'acceptabilité au-delà de l'attaque binaire ; le carnet [04b (base
de connaissances)](Argumentation-04b-Knowledge-Base-Python.ipynb) montre la
mémoire où les actes de ces protocoles déposent leurs arguments (jumeau C# :
[04b-CSharp](Argumentation-04b-Knowledge-Base-CSharp.ipynb)) ; le carnet [05
(vérification formelle)](Argumentation-05-Formal-Verification-Python.ipynb)
confie les verdicts à un vrai solveur.

**Statut et pérennité.** Les trois machines que ce carnet enseigne sont un
snapshot du retrait tronc #2137 (2026-09-17) ; le chemin de production est
`logic/dialogue_handler.py` (Tweety) — et c'est précisément ce chemin que ce
jumeau exécute, nativement, via le shade IKVM du dépôt. La distillation est
assumée comme contenu pédagogique : la machine à états y est enseignée en BCL
pur, et le dialogue de grounding réel y est exécuté sur le vrai moteur.

**Référence** — D. N. Walton, E. C. Krabbe (1995). *Commitment in Dialogue:
Basic Concepts of Interpersonal Reasoning*, SUNY Press — la typologie des six
types de dialogue et l'engagement (commitment) qui fonde les protocoles
reconstructeurs.
